# AgriLink — Price-Aware Recommendation

# AgriLink Customer Recommendation
This notebook is a standalone baseline. It runs with generated example data by default.
To use AgriLink data, set `USE_CSV = True` and update `CSV_PATH`. Expected columns are documented below.
Do not report results from synthetic data as real AgriLink performance.

**Goal:** rank available products using customer purchase preference and price relevance.
**Required real-data inputs:** order lines with `CustomerID`, `OrderID`, `Product`, `Quantity`, `UnitPrice`; plus a current product catalogue with `Product`, `CurrentPrice`, and optionally `Available`.
The example data is synthetic. The baseline estimates a customer's preferred price range from past paid unit prices and balances popularity with price-fit.

In [ ]:
import pandas as pd
import numpy as np

USE_CSV = False
ORDERS_CSV = "/content/agrilink_orders_with_prices.csv"
CATALOG_CSV = "/content/agrilink_product_catalog.csv"
rng = np.random.default_rng(21)
products = ["Apple","Banana","Beans","Brinjal","Cabbage","Carrot","Coriander",
            "Cucumber","Green Chilli","Onion","Orange","Papaya","Potato","Rice","Tomato"]
base_price = {p: float(rng.uniform(20, 160)) for p in products}
rows=[]
for c in range(1,81):
    chosen = rng.choice(products, size=7, replace=False)
    budget_factor = float(rng.choice([0.75, 1.0, 1.25]))
    for o in range(1,6):
        date = pd.Timestamp("2025-01-01") + pd.Timedelta(days=o*10)
        for p in rng.choice(chosen, size=3, replace=False):
            rows.append({"CustomerID":c,"OrderID":f"{c}_{o}","Product":p,
                         "Quantity":int(rng.integers(1,4)),
                         "UnitPrice":round(base_price[p]*budget_factor,2),
                         "OrderDate":date})
orders = pd.DataFrame(rows)
catalog = pd.DataFrame({"Product":products,
                        "CurrentPrice":[round(v,2) for v in base_price.values()],
                        "Available":True})
if USE_CSV:
    orders = pd.read_csv(ORDERS_CSV)
    catalog = pd.read_csv(CATALOG_CSV)
    missing_orders = {"CustomerID","OrderID","Product","Quantity","UnitPrice"} - set(orders.columns)
    missing_catalog = {"Product","CurrentPrice"} - set(catalog.columns)
    if missing_orders: raise ValueError(f"Orders CSV missing: {missing_orders}")
    if missing_catalog: raise ValueError(f"Catalog CSV missing: {missing_catalog}")
orders["UnitPrice"] = pd.to_numeric(orders["UnitPrice"], errors="coerce")
catalog["CurrentPrice"] = pd.to_numeric(catalog["CurrentPrice"], errors="coerce")
catalog = catalog.dropna(subset=["Product","CurrentPrice"])
if "Available" not in catalog.columns:
    catalog["Available"] = True
orders.head(), catalog.head()

## Recommendation scoring
We estimate the customer's typical paid unit price using a quantity-weighted median. Price fit decreases as the current price moves away from that preference. Product popularity supplies a simple relevance signal. The weights are configurable and should be validated experimentally.

In [ ]:
def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values, weights = values[mask], weights[mask]
    if len(values) == 0:
        return np.nan
    order = np.argsort(values)
    values, weights = values[order], weights[order]
    return values[np.searchsorted(np.cumsum(weights), weights.sum()/2)]

def price_aware_recommendations(customer_id, orders, catalog, top_n=10,
                                price_weight=0.5, popularity_weight=0.5,
                                exclude_purchased=False):
    hist = orders[orders["CustomerID"] == customer_id].copy()
    if hist.empty:
        preferred_price = orders["UnitPrice"].median()
    else:
        preferred_price = weighted_median(hist["UnitPrice"], hist["Quantity"])
    if not np.isfinite(preferred_price):
        preferred_price = catalog["CurrentPrice"].median()

    popularity = orders.groupby("Product")["OrderID"].nunique()
    result = catalog.copy()
    if "Available" in result.columns:
        result = result[result["Available"].astype(bool)]
    result["Popularity"] = result["Product"].map(popularity).fillna(0).astype(float)
    max_pop = result["Popularity"].max()
    result["PopularityScore"] = result["Popularity"] / max_pop if max_pop > 0 else 0.0

    # Relative price distance, bounded to [0,1]; robust when preferred price is zero.
    denom = max(abs(float(preferred_price)), 1.0)
    result["PriceFitScore"] = np.exp(
        -np.abs(result["CurrentPrice"] - preferred_price) / denom
    )
    total_weight = max(price_weight + popularity_weight, 1e-12)
    result["HybridPriceScore"] = (
        price_weight * result["PriceFitScore"] +
        popularity_weight * result["PopularityScore"]
    ) / total_weight

    if exclude_purchased and not hist.empty:
        bought = set(hist["Product"])
        result = result[~result["Product"].isin(bought)]

    result["EstimatedPreferredUnitPrice"] = preferred_price
    return result.sort_values("HybridPriceScore", ascending=False).head(top_n).reset_index(drop=True)

customer_id = orders["CustomerID"].iloc[0]
display(price_aware_recommendations(customer_id, orders, catalog))

## Compare price-fit and popularity contributions

In [ ]:
display(price_aware_recommendations(
    customer_id, orders, catalog, top_n=10,
    price_weight=0.8, popularity_weight=0.2
))
display(price_aware_recommendations(
    customer_id, orders, catalog, top_n=10,
    price_weight=0.2, popularity_weight=0.8
))

## Limitations
- This is a transparent heuristic, not a trained price-aware matrix-factorization model.
- Historical paid prices can reflect promotions, quantity discounts, and changing market prices; use transaction-time price and current price separately.
- Do not use price similarity as a substitute for availability, quality, or customer preference. Validate using chronological holdout data.